# BLOCO 1: O VOCABULÁRIO DAS LLMS

## Tokenização

In [ ]:
# Exemplo com tiktoken (O tokenizador do GPT)

In [ ]:
!pip install tiktoken

In [ ]:
import tiktoken

encoding = tiktoken.get_encoding("gpt2")

texto = "The puppy and the kitten played in the street all day long."
tokens = encoding.encode(texto)

print("Tokens (ids):", tokens)
print("Quantidade:", len(tokens))
print("Decodificado:", [encoding.decode([t]) for t in tokens])

Tokens (ids): [464, 26188, 290, 262, 42143, 2826, 287, 262, 4675, 477, 1110, 890, 13]
Quantidade: 13
Decodificado: ['The', ' puppy', ' and', ' the', ' kitten', ' played', ' in', ' the', ' street', ' all', ' day', ' long', '.']


pontos de destaque:
* "equipe" virou dois tokens, isso certamente acontece por que "ipe" é um sufixo comum no dataset no qual o tokenizador foi treinado.
* O espaço é importante na tokenização, " de" é diferente de "de"

Vocabulário: conjunto de todos os tokens possíveis. O GPT-2 tem 50.257 tokens. O GPT-4 tem ~100.000.

# BLOCO 2: DATALOADER

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
  """
  Constrói um dataset de linguagem com janela deslizante.
  Cada item é um par (entrada, alvo)
  onde alvo = entrada deslocada em 1.
  """
  def __init__(self, text, tokenizer, context_length, stride = False):
    if not stride:
      stride = context_length
    self.input_ids = []
    self.target_ids = []

    token_ids = tokenizer.encode(text)

    for i in range(0, len(token_ids) - context_length, stride):
      entrada = token_ids[i : i + context_length]
      alvo = token_ids[i+1 : i + context_length + 1]

      self.input_ids.append(torch.tensor(entrada))
      self.target_ids.append(torch.tensor(alvo))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]



In [ ]:
texto = "The puppy and the kitten played in the street all day long."

dataset = TextDataset(texto, encoding, context_length= 8, stride = 4)
dataloader = DataLoader(dataset, batch_size = 4, shuffle = True)

# Inspecionando um batch
X_batch, y_batch = next(iter(dataloader))
print("Shape de X:", X_batch.shape)   # (4, 16) — 4 sequências de 16 tokens
print("Shape de y:", y_batch.shape)   # (4, 16) — os alvos correspondentes


# Verificando o deslocamento
print("\nEntrada (tokens):", X_batch[0][:6].tolist())
print("Alvo    (tokens):", y_batch[0][:6].tolist())
# Alvo deve ser a entrada deslocada em 1 posição

Shape de X: torch.Size([2, 8])
Shape de y: torch.Size([2, 8])

Entrada (tokens): [42143, 2826, 287, 262, 4675, 477]
Alvo    (tokens): [2826, 287, 262, 4675, 477, 1110]


In [ ]:
print("\nEntrada (texto):", encoding.decode(X_batch[0][:6].tolist()))
print("Alvo    (texto):", encoding.decode(y_batch[0][:6].tolist()))


Entrada (texto):  kitten played in the street all
Alvo    (texto):  played in the street all day


# BLOCO 3: EMBEDDINGS

In [ ]:
import torch.nn as nn

vocab_size = 50257 # tamanho do vocabulário do GPT2
embedding_dim = 256

# A tabela de embeddings é inicialmente iniciada de forma aleatória
embedding = nn.Embedding(vocab_size, embedding_dim)

print("Shape da tabela:", embedding.weight.shape)

token_ids = torch.tensor([5234, 8801, 1234])   # "gato", "cachorro"
vetores = embedding(token_ids)

print("Shape dos vetores:", vetores.shape)   # (3, 256)
print("Vetor do primeiro token:", vetores[0][:8])


Shape da tabela: torch.Size([50257, 256])
Shape dos vetores: torch.Size([3, 256])
Vetor do primeiro token: tensor([-0.3132,  1.5911, -1.1604, -1.9863, -0.7526,  0.5729, -0.6535, -0.1865],
       grad_fn=<SliceBackward0>)


In [ ]:
# Uma sequência de tokens (1 frase tokenizada)
frase = "O gato sentou no tapete"
token_ids = torch.tensor(encoding.encode(frase))   # ex: [46, 5234, 3332, ...]

print("Tokens:", token_ids)
print("Quantidade de tokens:", len(token_ids))

# Embedding da sequência inteira
vetores = embedding(token_ids)
print("Shape:", vetores.shape)   # (n_tokens, 256)

Tokens: tensor([   46,   308,  5549,  1908,   280,   645,  9814, 14471])
Quantidade de tokens: 8
Shape: torch.Size([8, 256])


## Embedding posicional

In [ ]:
context_length = 16

token_embedding = nn.Embedding(vocab_size, embedding_dim)

pos_embedding = nn.Embedding(context_length, embedding_dim)

# Sequência de exemplo
tokens = torch.tensor(encoding.encode("The cat sit on the carpet"))[:context_length]
n = len(tokens)

# Posições: 0, 1, 2, ..., n-1
posicoes = torch.arange(n)   # tensor([0, 1, 2, 3, 4])

# Embedding final = embedding do token + embedding da posição
x = token_embedding(tokens) + pos_embedding(posicoes)
print("Shape final:", x.shape)   # (5, 256)

Shape final: torch.Size([6, 256])


# BLOCO 4: ATENÇÃO

A fórmula da atenção:
Attention(Q, K, V) = softmax( Q · Kᵀ / √d_k ) · V

In [ ]:
import torch.nn.functional as F
import math

seq_len = 5    # 5 tokens na sequência
d_model = 16   # dimensão dos embeddings
d_k = 8        # dimensão das queries e keys

# Entrada: sequência de embeddings (simulada)
x = torch.randn(seq_len, d_model)   # (5, 16)

# Matrizes de projeção — aprendíveis no modelo real
W_Q = torch.randn(d_model, d_k)
W_K = torch.randn(d_model, d_k)
W_V = torch.randn(d_model, d_k)

# Passo 1: projetar x em Q, K, V
Q = x @ W_Q   # (5, 8) — queries
K = x @ W_K   # (5, 8) — keys
V = x @ W_V   # (5, 8) — values

print("Shape de Q:", Q.shape)
print("Shape de K:", K.shape)
print("Shape de V:", V.shape)

Shape de Q: torch.Size([5, 8])
Shape de K: torch.Size([5, 8])
Shape de V: torch.Size([5, 8])


In [ ]:
# Passo 2: calcular scores de atenção
# Quão compatível é cada query com cada key?
scores = Q @ K.T   # (5, 5) — score de cada par (token_i, token_j)

print("Shape dos scores:", scores.shape)
print("Scores brutos:\n", scores.round(decimals=2))

Shape dos scores: torch.Size([5, 5])
Scores brutos:
 tensor([[  2.0300, -29.0300, -11.5800,  -5.3300, -10.8400],
        [-25.4600,  27.2700, -13.1300,  -6.3600,  27.2500],
        [ 29.6400,  24.8700,  37.1300,  20.9500,   0.5700],
        [-15.0000, -53.6000, -43.6800,   7.1900,  -7.1900],
        [-20.1500, -14.4500, -15.4200,  10.0600, -15.4100]])


a matriz de scores tem shape (5, 5). O elemento [i, j] representa o quão relevante o token j é para o token i. É uma matriz de "atenção" entre todos os pares de tokens.

In [ ]:
# Passo 3: escalar pelo fator √d_k
# Por que escalar? Produtos internos crescem com a dimensão.
# Sem escala, os valores ficam grandes demais → softmax satura → gradientes somem.
scores_scaled = scores / math.sqrt(d_k)

# Passo 4: softmax — normaliza os scores em distribuição de probabilidade
# Cada linha soma 1: o token i distribui 100% de sua atenção entre todos os tokens
weights = F.softmax(scores_scaled, dim=-1)   # (5, 5)

print("Pesos de atenção (linha 0):", weights[0].round(decimals=3))
print("Soma da linha 0:", weights[0].sum().item())   # deve ser 1.0

Pesos de atenção (linha 0): tensor([0.9150, 0.0000, 0.0070, 0.0680, 0.0100])
Soma da linha 0: 1.0000001192092896


In [ ]:
# Passo 5: combinar os values usando os pesos
# Para cada token, soma ponderada de todos os values
output = weights @ V   # (5, 8)

print("Shape do output:", output.shape)
print("\nCada token agora carrega informação contextualizada de toda a sequência.")

Shape do output: torch.Size([5, 8])

Cada token agora carrega informação contextualizada de toda a sequência.


o output tem o mesmo shape que V mas agora o vetor de cada token é uma mistura de todos os outros tokens, ponderada pela relevância calculada. O token "banco" num contexto financeiro vai ter um vetor diferente do que num contexto de sentar, porque os pesos de atenção vão ser diferentes.

## Causal Masking

In [ ]:
seq_len = 5
mask = torch.tril(torch.ones(seq_len, seq_len))
print("Máscara causal:\n", mask)

scores_masked = scores_scaled.masked_fill(mask == 0, float('-inf'))

weights_causal = F.softmax(scores_masked, dim=-1)
print("\nPesos causais (linha 2 — token vê só posições 0, 1, 2):")
print(weights_causal[2].round(decimals=3))

Máscara causal:
 tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

Pesos causais (linha 2 — token vê só posições 0, 1, 2):
tensor([0.0650, 0.0120, 0.9230, 0.0000, 0.0000])


GPT e modelos decoder-only usam exatamente essa máscara. Durante a geração de texto, o modelo produz um token por vez — e a máscara garante que durante o treinamento ele aprendeu a fazer isso honestamente.

# BLOCO 5: ARQUITETURA TRANSFORMER

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, n_heads):
    super().__init__()
    assert d_model % n_heads == 0

    self.n_heads = n_heads
    self.d_k = d_model // n_heads

    self.W_Q = nn.Linear(d_model, d_model)
    self.W_K = nn.Linear(d_model, d_model)
    self.W_V = nn.Linear(d_model, d_model)
    self.W_O = nn.Linear(d_model, d_model)

  def forward(self, x):
    B,T,C = x.shape # batch, seq_len, d_model

    Q = self.W_Q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
    K = self.W_K(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
    V = self.W_V(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)

    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)

    # Máscara causal
    mask = torch.tril(torch.ones(T, T, device=x.device))
    scores = scores.masked_fill(mask == 0, float('-inf'))

    weights = F.softmax(scores, dim=-1)
    out = weights @ V   # (B, n_heads, T, d_k)

    # Juntar as cabeças
    out = out.transpose(1, 2).contiguous().view(B, T, C)
    return self.W_O(out)



.view() e .transpose() são operações de reshape para separar e juntar as cabeças. O conteúdo matemático é idêntico ao que implementamos anteriormente, só que paralelizado

In [ ]:
class TransformerBlock(nn.Module):
  def __init__(self,d_model, n_heads, ffn_dim, dropout = 0.1):
    super().__init__()

    self.attn = MultiHeadAttention(d_model, n_heads)
    self.ffn = nn.Sequential(
        nn.Linear(d_model, ffn_dim),
        nn.GELU(), # variante suave da ReLU — padrão em LLMs
        nn.Linear(ffn_dim,d_model)
    )
    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    # Atenção com conexão residual + normalização (Pre-Norm — padrão moderno)
    x = x + self.dropout(self.attn(self.norm1(x)))
    # FFN com conexão residual + normalização        x = x + self.dropout(self.ffn(self.norm2(x)))
    return x

# Testando um bloco
block = TransformerBlock(d_model=64, n_heads=4, ffn_dim=256)
x = torch.randn(2, 16, 64)   # batch=2, seq_len=16, d_model=64
out = block(x)
print("Shape de entrada:", x.shape)
print("Shape de saída:  ", out.shape)   # idêntico — blocos podem ser empilhados


Shape de entrada: torch.Size([2, 16, 64])
Shape de saída:   torch.Size([2, 16, 64])


## Um modelo completo estilo GPT

Token ids
    → token_embedding + positional_embedding
    → N × TransformerBlock
    → LayerNorm final
    → Linear(d_model, vocab_size)   ← "cabeça de linguagem"
    → logits sobre o vocabulário

In [ ]:
class MiniGPT(nn.Module):
  def __init__(self, vocab_size, context_length, d_model, n_heads, n_layers, ffn_dim):
    super().__init__()
    self.token_emb = nn.Embedding(vocab_size, d_model)
    self.pos_emb = nn.Embedding(context_length, d_model)

    self.blocks = nn.Sequential(*[
        TransformerBlock(d_model, n_heads, ffn_dim)
        for _ in range(n_layers)
    ])

    self.norm = nn.LayerNorm(d_model)
    self.lm_head = nn.Linear(d_model,vocab_size, bias=False)

  def forward(self,token_ids):
    B, T = token_ids.shape

    # Embeddings
    tok = self.token_emb(token_ids)
    pos = self.pos_emb(torch.arange(T, device=token_ids.device))
    x = tok + pos                 # (B, T, d_model)

    x = self.blocks(x)
    x = self.norm(x)

    logits = self.lm_head(x)

    return logits

In [ ]:
# Instanciando um modelo pequeno
model = MiniGPT(
    vocab_size=50257,
    context_length=256,
    d_model=128,
    n_heads=4,
    n_layers=4,
    ffn_dim=512,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Parâmetros totais: {total_params:,}")   # ~13 milhões

# Forward pass
tokens = torch.randint(0, 50257, (2, 32))   # batch=2, seq=32
logits = model(tokens)
print("Shape dos logits:", logits.shape)     # (2, 32, 50257)

Parâmetros totais: 13,691,904
Shape dos logits: torch.Size([2, 32, 50257])


cada posição da sequência produz uma distribuição sobre o vocabulário inteiro — o modelo está prevendo qual token vem a seguir em cada posição. Durante o treinamento, comparamos isso com o token real (usando cross-entropy). Durante a geração, amostramos desta distribuição.